In [1]:
import numpy as np
import zarr

from tqdm  import tqdm

In [2]:
from zarro import (
    DataKind,
    WriteType,
    ControlVar,
    AxisSpec,
    ReadoutSpec,
    MeasurementSchema,
    StreamWriter
)

## **Examples of 2D scans**

In [3]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))
Vgt = ControlVar(name="Vgt", values=np.linspace(1, 2, 10, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpec([Vgt]),
        AxisSpec([Vch, Vm])
    ],
    measure = [
        ReadoutSpec(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpec(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpec(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24"

### **2D scan / save all at once in the end of scan**

use `writer.write_all(dict)`

In [15]:
mschema.writetype = WriteType.ALL

writer = StreamWriter(
    root=root_path,
    experiment_id="exp_007",
    schema=mschema,
    overwrite=True,
)

print(writer._dataset_map["lockin_X"].info)
print("********************")
print(writer._dataset_map["S21"].info)

Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200)
Chunk shape        : (10, 200)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 8000 (7.8K)
********************
Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200, 1601)
Chunk shape        : (10, 200, 1601)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 12808000 (12.2M)


In [6]:
MAG_All = []
LOCKIN_All = []
TEMP_All = []

for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    LOCKIN_All.append(LOCKIN)
    TEMP_All.append(TEMP)
    MAG_All.append(MAG)

writer.write_all({"S21": MAG_All, "temperature": TEMP_All, "lockin_X": LOCKIN_All})
writer.add_array("Freqs", freq, meta={"unit": "Hz", "description": "Frequency points for S21 measurement"})

100%|██████████| 10/10 [00:00<00:00, 164.78it/s]


### **2D scan / save trace at each step**

use `writer.write_trace((i,), dict)`

In [7]:
mschema.writetype = WriteType.TRACEWISE

writer = StreamWriter(
    root=root_path,
    experiment_id="exp_007",
    schema=mschema,
    overwrite=True,
)

print(writer._dataset_map["lockin_X"].info)
print("********************")
print(writer._dataset_map["S21"].info)

Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200)
Chunk shape        : (1, 200)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 8000 (7.8K)
********************
Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200, 1601)
Shard shape        : (5, 200, 1601)
Chunk shape        : (1, 200, 1601)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 12808000 (12.2M)


In [8]:
for i in tqdm(range(len(Vgt.values))):
    MAG = []
    LOCKIN = []
    TEMP = []
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()

        MAG.append(mag)
        LOCKIN.append(lockinx)
        TEMP.append(T1K)

    writer.write_trace((i,), {"S21": MAG, "temperature": TEMP, "lockin_X": LOCKIN})

100%|██████████| 10/10 [00:00<00:00, 82.15it/s]


### **2D scan / save trace at each point**

use `writer.write_point((i,j), dict)`

if the writing is slow, try to adjust `writer.target_file_size_bytes` in a way, that `shard_shape` is as close to `chunk_shape`

In [12]:
mschema.writetype = WriteType.POINTWISE

writer = StreamWriter(
    root=root_path,
    experiment_id="exp_007",
    schema=mschema,
    target_file_size_bytes=1*1024*1024,  # 5 MB
    overwrite=True,
)


print(writer._dataset_map["lockin_X"].info)
print("********************")
print(writer._dataset_map["S21"].info)

Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200)
Chunk shape        : (1, 1)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 8000 (7.8K)
********************
Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (10, 200, 1601)
Shard shape        : (1, 166, 1601)
Chunk shape        : (1, 1, 1601)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 12808000 (12.2M)


In [13]:
for i in tqdm(range(len(Vgt.values))):
    for j in range(len(Vch.values)):
        freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
        lockinx = np.random.random()
        T1K = np.random.random()
        writer.write_point((i, j), {"S21": mag, "temperature": T1K, "lockin_X": lockinx})

100%|██████████| 10/10 [00:09<00:00,  1.06it/s]


## **Examples of 1D scans**

In [16]:
Vch = ControlVar(name="Vch", values=np.linspace(0, 1, 200, dtype=np.float32))
Vm  = ControlVar(name="Vm",  values=np.logspace(np.log10(0.1), np.log10(10), 200, dtype=np.float32))

mschema = MeasurementSchema(
    control = [
        AxisSpec([Vch, Vm])
    ],
    measure = [
        ReadoutSpec(name="lockin_X", kind=DataKind.SCALAR, unit="V"),
        ReadoutSpec(name="S21", kind=DataKind.TRACE, shape=(1601,), unit="dBm"),
        ReadoutSpec(name="temperature", kind=DataKind.SCALAR, unit="K"),
    ],
    writetype=WriteType.ALL,
)

root_path = "/Users/helium/Documents/data/2026-02-24"

### **1D scan / save all at once in the end of scan**

use `writer.write_all(dict)`

In [21]:
mschema.writetype = WriteType.ALL

writer = StreamWriter(
    root=root_path,
    experiment_id="exp_007",
    schema=mschema,
    overwrite=True,
)

print(writer._dataset_map["lockin_X"].info)
print("********************")
print(writer._dataset_map["S21"].info)

Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (200,)
Chunk shape        : (200,)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 800
********************
Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (200, 1601)
Chunk shape        : (200, 1601)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 1280800 (1.2M)


In [22]:
MAG = []
LOCKIN = []
TEMP = []
for j in range(len(Vch.values)):
    freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
    lockinx = np.random.random()
    T1K = np.random.random()

    MAG.append(mag)
    LOCKIN.append(lockinx)
    TEMP.append(T1K)

writer.write_all({"S21": MAG, "temperature": TEMP, "lockin_X": LOCKIN})

### **1D scan / save data at each point**

use `writer.write_point((i,), dict)`

if the writing is slow, try to adjust `writer.target_file_size_bytes` in a way, that `shard_shape` is as close to `chunk_shape`

In [19]:
mschema.writetype = WriteType.POINTWISE

writer = StreamWriter(
    root=root_path,
    experiment_id="exp_007",
    schema=mschema,
    target_file_size_bytes=1*1024*1024,  # 5 MB
    overwrite=True,
)


print(writer._dataset_map["lockin_X"].info)
print("********************")
print(writer._dataset_map["S21"].info)

Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (200,)
Chunk shape        : (1,)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 800
********************
Type               : Array
Zarr format        : 3
Data type          : Float32(endianness='little')
Fill value         : 0.0
Shape              : (200, 1601)
Shard shape        : (200, 1601)
Chunk shape        : (1, 1601)
Order              : C
Read-only          : False
Store type         : LocalStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 1280800 (1.2M)


In [20]:
for j in range(len(Vch.values)):
    freq, mag, phase = np.random.random(1601), np.random.random(1601), np.random.random(1601)
    lockinx = np.random.random()
    T1K = np.random.random()

    writer.write_point((j,), {"S21": mag, "temperature": T1K, "lockin_X": lockinx})

## **Reading data**

In [23]:
z2 = zarr.open(root_path + "/exp_007" + '/vault.zarr', mode='r')

In [24]:
z2.info

Name        : 
Type        : Group
Zarr format : 3
Read-only   : True
Store type  : LocalStore

In [25]:
z2["S21"][:,159]

array([6.6778582e-01, 3.5440177e-01, 7.2138220e-01, 9.2517263e-01,
       9.6445692e-01, 6.9100505e-01, 2.1918242e-01, 4.9664122e-01,
       4.8838314e-01, 3.1949875e-01, 8.0294915e-02, 6.0467232e-02,
       3.6437729e-01, 7.1964419e-01, 7.5735760e-01, 5.8966082e-01,
       9.3991178e-01, 9.2177165e-01, 1.8685554e-01, 2.3022050e-01,
       9.9667525e-01, 7.6479816e-01, 8.8859725e-01, 7.9501683e-01,
       8.6234593e-01, 8.3521187e-01, 5.2777278e-01, 4.2799097e-01,
       2.5605425e-01, 8.4967661e-01, 2.1124874e-01, 2.1114087e-01,
       8.0694979e-01, 8.0615300e-01, 8.3765686e-02, 7.0206910e-01,
       2.1348067e-01, 5.1887530e-01, 2.0151237e-01, 6.0489142e-01,
       9.5476097e-01, 7.5540400e-01, 4.5600826e-01, 9.1512316e-01,
       3.2391140e-01, 3.3110020e-01, 5.8078462e-01, 9.2471600e-01,
       2.6159674e-01, 6.2927765e-01, 9.9084586e-01, 3.0290839e-01,
       9.4597565e-04, 8.5857821e-01, 8.4690601e-01, 3.5739613e-01,
       5.1118988e-01, 4.2331412e-01, 8.5226160e-01, 3.1691563e